### Dataset

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import sys
import numpy as np
import torch
from sklearn.manifold import TSNE
import yaml
import matplotlib.pyplot as plt

sys.path.append(os.path.abspath("../src"))
sys.path.append(os.path.abspath("../src/losses"))
sys.path.append(os.path.abspath("../models"))

from dataset import get_data_loaders
from backbone import BackBone
from supcon import SupConLoss
from utils import deterministic, backbone_train

In [3]:
SEED = 42

In [4]:
deterministic(SEED)
dataloaders = get_data_loaders(batch_size=512)

/home/alumno1/miniconda3/envs/vision/lib/python3.11/site-packages/torchvision/datasets/cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


Task [0, 1]: Train=9000, Val=1000, Test=2000
Task [2, 3]: Train=9000, Val=1000, Test=2000
Task [4, 5]: Train=9000, Val=1000, Test=2000
Task [6, 7]: Train=9000, Val=1000, Test=2000
Task [8, 9]: Train=9000, Val=1000, Test=2000


In [5]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)
backbone = BackBone().to(device)
optimizer = torch.optim.Adam(backbone.parameters(), lr=1e-2)
criterion = SupConLoss(tau=0.5)
tsne = TSNE(n_components=2, random_state=SEED)

cuda


In [6]:
deterministic(SEED)

epochs = 50

history, emb_labels = backbone_train(backbone, dataloaders[0][0], optimizer, criterion, "backbone", tsne, epochs)

Epochs: 100%|██████████| 50/50 [01:16<00:00,  1.53s/epoch, loss=5.64]


In [7]:

tsne_path = "../results/backbone_tsne.npz"
os.makedirs(os.path.dirname(tsne_path), exist_ok=True)

tsne_embeddings = np.stack([emb_2d for emb_2d, _ in emb_labels], axis=0)
tsne_labels = np.stack([y for _, y in emb_labels], axis=0)
snapshot_epochs = np.array([0, epochs // 2, epochs], dtype=np.int32)[:len(emb_labels)]
np.savez_compressed(
    tsne_path,
    embeddings=tsne_embeddings,
    labels=tsne_labels,
    epochs=snapshot_epochs,
)

path = "../results/backbone_history.yaml"
os.makedirs(os.path.dirname(path), exist_ok=True)

with open(path, "w") as f:
    yaml.dump(history, f)